# Session 4 — Solutions

Worked solutions for weights, systematics, fitting, goodness of fit, and limit calculation. Requires processor output (output_2017.pkl or output_2017_full.pkl) from run_analysis.py.

MC histograms from `run_analysis.py --full` are already luminosity-normalised, so do not apply an additional xsec×lumi scaling here.

## Load results and build data / background histograms

In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
# Needed so we can import the project modules when running from `solutions/`.
sys.path.append("..")

# Avoid recursion limit issues when unpickling nested accumulator/hist objects
try:
    sys.setrecursionlimit(8000)
except Exception:
    pass

import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import poisson, chi2

import os

def _load_results():
    """Load the processor output produced by `run_analysis.py`.

    We try a few common paths so this works whether you run from the repo root
    or from inside the `solutions/` directory.
    """
    candidates = [
        "output/output_2017.pkl", "output/output_2017_full.pkl",
        "output_2017.pkl", "output_2017_full.pkl",
        "../output/output_2017.pkl", "../output/output_2017_full.pkl",
    ]
    for path in candidates:
        if os.path.isfile(path):
            return pickle.load(open(path, "rb"))
    raise FileNotFoundError("Run run_analysis.py first. Outputs go to output/.")

bundle_or_results = _load_results()
if isinstance(bundle_or_results, dict) and isinstance(bundle_or_results.get("samples"), dict):
    metadata = bundle_or_results.get("metadata", {})
    results = bundle_or_results["samples"]
else:
    metadata = {}
    results = bundle_or_results

from config.datasets_2017 import data_and_bkg_keys

data_datasets, bkg_datasets = data_and_bkg_keys(results)
print("Data keys:", data_datasets, "| Bkg keys (excl. signal):", bkg_datasets)

# Main observable: prefer cos(theta*) in SR; fall back to MET in SR
# (This keeps the script working even if cos(theta*) wasn't produced.)
def _unwrap_hist_obj(obj):
    return obj._hist if hasattr(obj, "_hist") else obj

def get_main_hist(h):
    # Structured bundle: prefer SR projection from region-aware histograms.
    hbr = h.get("hists_by_region", {}) if isinstance(h, dict) else {}
    if isinstance(hbr, dict):
        cts = _unwrap_hist_obj(hbr.get("cos_theta_star"))
        if cts is not None:
            try:
                return cts[{"region": "sr"}]
            except Exception:
                pass
        recoil = _unwrap_hist_obj(hbr.get("recoil"))
        if recoil is not None:
            try:
                return recoil[{"region": "sr"}]
            except Exception:
                pass
    # Legacy flat format fallback.
    cts = _unwrap_hist_obj(h.get("cos_theta_star"))
    if cts is not None:
        return cts
    recoil = _unwrap_hist_obj(h.get("recoil"))
    if recoil is not None:
        return recoil
    raise KeyError("No compatible main histogram found (cos_theta_star/recoil).")


def sum_hists(results, names):
    """Sum per-dataset 1D histograms into one histogram."""
    out = None
    for n in names:
        if n not in results:
            continue
        h = get_main_hist(results[n])
        if out is None:
            out = h.copy()
        else:
            out = out + h
    return out

# Build total data and total background histograms
data_hist = sum_hists(results, data_datasets)
bkg_hist = sum_hists(results, bkg_datasets)

# Convert histogram counts into flat NumPy arrays for fitting
if data_hist is not None:
    obs = np.asarray(data_hist.values()).flatten()
    obs = np.maximum(obs, 0)
else:
    obs = None

if bkg_hist is not None:
    bkg = np.asarray(bkg_hist.values()).flatten()
    # Avoid zeros in denominators/log-likelihood
    bkg = np.maximum(bkg, 1e-6)
    if obs is None:
        obs = np.zeros(len(bkg))
else:
    # Fallback (shouldn't happen if results are present)
    bkg = np.ones(25) * 10
    if obs is None:
        obs = np.zeros(25)

print("Data sum:", obs.sum(), "Bkg sum:", bkg.sum(), "(main observable: cos_theta_star if present else recoil)")

## Binned fit (one scale factor for background)

In [ ]:
# Negative log-likelihood (Poisson) for a simple background-only model
# We fit a *single* scale factor that multiplies the total background prediction.
def nll(scale):
    lam = scale * bkg
    # Use a small floor to avoid log(0) for empty bins
    return -np.sum(poisson.logpmf(obs.astype(int), np.maximum(lam, 1e-10)))

# Best-fit scale factor for background normalization
fit = minimize(nll, x0=[1.0], bounds=[(0.01, 10)])
scale_best = fit.x[0]
expected = scale_best * bkg
print("Best-fit background scale:", scale_best)
print("Total observed:", obs.sum(), "Total expected (after fit):", expected.sum())

## Goodness of fit

In [ ]:
# Goodness of fit: a simple Pearson chi^2 test against the fitted expectation
chi2_val = np.sum((obs - expected) ** 2 / np.where(expected > 0, expected, 1))
ndof = len(obs) - 1  # minus 1 fitted parameter (the background scale)
pvalue = 1 - chi2.cdf(chi2_val, ndof)
print(f"chi2 = {chi2_val:.2f}, ndof = {ndof}, p-value = {pvalue:.4f}")

# Pulls highlight which bins drive discrepancies: (data - expectation)/sqrt(expectation)
pulls = (obs - expected) / np.sqrt(np.where(expected > 0, expected, 1))
plt.figure(figsize=(8, 3))
plt.bar(range(len(pulls)), pulls, edgecolor="black", alpha=0.7)
plt.xlabel("Bin")
plt.ylabel("Pull")
plt.title("Pull distribution")
plt.axhline(0, color="gray", ls="--")
plt.tight_layout()
plt.show()

## Simple 95% CL upper limit (single bin: total SR yield)

In [ ]:
# Simple 95% CL upper limit (single bin: total SR yield)
# Here we collapse all bins into a single Poisson counting experiment.
# This is *not* a full profile-likelihood/CLs limit, but it demonstrates the idea.

# One-bin: observed = total data; background = total fitted expectation
n_obs = int(obs.sum())
b = expected.sum()

# Find s_95 such that P(N > n_obs | b + s_95) = 0.05 (one-sided)
from scipy.stats import poisson

def p_exceed(s):
    return 1 - poisson.cdf(n_obs, b + s)

s_vals = np.linspace(0, 50, 200)
p_vals = [p_exceed(s) for s in s_vals]
idx = np.argmin(np.abs(np.array(p_vals) - 0.05))
s_95 = s_vals[idx]
print(f"Approximate 95% CL upper limit on signal events (single bin): s_95 ≈ {s_95:.1f}")